# Conversation Analysis — Summarization and Intent Classification with FLAN-T5

Conversations are noisy: agents go off-topic, callers repeat themselves, context gets buried.
This notebook shows how a single encoder-decoder model (FLAN-T5) can be prompted to extract
key points from a transcript *and* classify the speaker's intent — without fine-tuning.

| Step | Concept | Key Idea |
|------|---------|---------|
| 1 | Why automated analysis? | Keyword counting misses paraphrase; models capture meaning |
| 2 | FLAN-T5 architecture | Encoder-decoder + instruction tuning: one model, many tasks |
| 3 | Summarization pipeline | `pipeline("summarization")` under the hood |
| 4 | Live demo | Summarize a customer-service transcript |
| 5 | Your turn | Change the prompt style; observe how output shifts |
| Summary | Key insights | Consolidated takeaways |


In [ ]:
# ── Install dependencies (skip if already present) ─────────────────────────────
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

_ensure("transformers")
_ensure("torch")
_ensure("sentencepiece")  # required by T5 tokenizer
print("Dependencies ready.")


In [ ]:
# ── Imports and deterministic seed ─────────────────────────────────────────────
import torch
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Torch version : {torch.__version__}")
print(f"Device        : {'cuda' if torch.cuda.is_available() else 'cpu'}")


---

## Part 1 — Why does keyword counting fail?

The oldest approach to conversation analysis is to count domain keywords:
if the transcript contains "refund", classify it as a billing complaint.
This breaks down as soon as a customer says "I don't want a refund, I need a replacement."

Predict before you run:
> If you count the word "refund" in the sentence above, what label do you assign?
> Does that match the customer's actual intent?

Run the cell below to see the failure.


In [ ]:
# ── Naive keyword-counting classifier ──────────────────────────────────────────
# The simplest possible "NLP": count domain words and pick the majority.

KEYWORDS = {
    "billing":     ["refund", "charge", "invoice", "payment"],
    "technical":   ["error", "crash", "bug", "slow", "broken"],
    "shipping":    ["delivery", "package", "tracking", "delay"],
}

SAMPLE = (
    "I don't want a refund — I just need you to fix the broken tracking "
    "on my package. The delivery estimate keeps changing and I'm getting "
    "error messages every time I log in to check."
)

def keyword_classify(text):
    counts = {topic: sum(text.lower().count(w) for w in words)
              for topic, words in KEYWORDS.items()}
    return max(counts, key=counts.get), counts

label, counts = keyword_classify(SAMPLE)
print("Transcript:")
print(f"  {SAMPLE}")
print()
print("Keyword hit counts:", counts)
print(f"  -> Predicted intent: {label}")
print("  -> Actual intent: shipping (with a technical sub-issue)")
print("  -> The refund keyword count is 1, so billing wins — wrong.")


#### What just happened — and what's missing

The keyword counter assigned *billing* because "refund" appears once, even though the
customer explicitly says they do *not* want a refund.
Keyword matching is bag-of-words: it cannot negate, paraphrase, or compose.

A model that reads the sentence as a sequence — not a bag — can resolve the negation.
That is what FLAN-T5 does.


---

## Part 2 — FLAN-T5: encoder-decoder + instruction tuning

FLAN-T5 is the T5 model family fine-tuned on a large collection of *instruction-formatted*
tasks (FLAN = Finetuned LAnguage Net).

**Architecture** (encoder-decoder):

$$\text{output} = \text{Decoder}(\text{Encoder}(x_{\text{input}}))$$

Plain-English gloss: the encoder reads the full input and builds a rich context vector;
the decoder generates the output token by token, attending to that context.
Unlike a decoder-only model, the encoder can see all input tokens bidirectionally
before any output is produced — ideal for tasks like summarization where understanding
the whole input matters.

**Instruction tuning** means the model was trained on prompts of the form
`"Summarize: <text>"` or `"What is the intent of: <text>"`.
You steer the model by writing a natural-language prefix — no gradient updates needed.

| Property | FLAN-T5-small | FLAN-T5-base |
|----------|-------------|--------------|
| Parameters | 77 M | 250 M |
| RAM (FP32) | ~300 MB | ~1 GB |
| Speed (CPU) | fast | moderate |
| Best for | prototyping | quality |


In [ ]:
# ── Load FLAN-T5-small (downloads ~300 MB on first run) ─────────────────────────
# We use the raw tokenizer + model to see what's happening inside the pipeline.

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"
print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()

print(f"  -> Encoder layers : {model.config.num_layers}")
print(f"  -> d_model        : {model.config.d_model}")
print(f"  -> Decoder layers : {model.config.num_decoder_layers}")
print(f"  -> Vocab size     : {model.config.vocab_size:,}")
print("Model ready.")


#### What just happened — and what's missing

We loaded FLAN-T5-small with its encoder and decoder stacks.
The model accepts a *text prompt* as input and generates *text* as output —
the task is entirely encoded in the prompt string we hand it.

Next we build the actual functions: one for key-point extraction, one for intent
classification. Both use the same model; only the prompt prefix changes.


---

## Part 3 — Implementation: prompt-driven summarization and classification

The design is simple:
- **Summarization prompt**: `"List the key points from this conversation: <text>"`
- **Intent prompt**: `"Classify the intent of the following customer message into one of [billing, technical, shipping, general]: <text>"`

Long texts are split into 400-word chunks; each chunk is summarized separately, then
the chunk summaries are joined and summarized once more.


In [ ]:
# ── extract_key_points: chunk-aware summarizer ─────────────────────────────────

def _generate(prompt: str, max_new_tokens: int = 200) -> str:
    """Run one forward pass through encoder-decoder and return decoded text."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # greedy for reproducibility
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def extract_key_points(text: str, chunk_words: int = 300) -> str:
    """Summarize a conversation transcript, handling long inputs via chunking."""
    words = text.split()
    if len(words) <= chunk_words:
        prompt = f"List the key points from this conversation: {text}"
        return _generate(prompt)

    # Split into overlapping chunks and summarize each
    chunks = [" ".join(words[i : i + chunk_words])
              for i in range(0, len(words), chunk_words)]
    summaries = []
    for i, chunk in enumerate(chunks):
        prompt = f"summarize: {chunk}"
        summaries.append(_generate(prompt, max_new_tokens=100))
        print(f"  Chunk {i+1}/{len(chunks)} done.")

    combined = " ".join(summaries)
    return _generate(f"List the key points from this text: {combined}")


def classify_intent(text: str) -> str:
    """Classify customer intent with an instruction prompt."""
    labels = "[billing, technical, shipping, general]"
    prompt = (f"Classify the intent of the following customer message into one of "
              f"{labels}: {text}")
    return _generate(prompt, max_new_tokens=20)

print("Helper functions defined.")


#### What just happened — and what's missing

`extract_key_points` and `classify_intent` both call the same `_generate` helper.
The *only* difference between them is the prompt prefix — FLAN-T5's instruction
tuning means the prefix is enough to steer the decoder toward a different output style.

For very long transcripts the chunking strategy introduces a seam: chunk summaries
are concatenated before the final pass, so cross-chunk context can be lost.
This is a known limitation of pipeline-style abstractive summarization.


---

## Part 4 — Live demo

We run both functions on the same difficult sample from Part 1 — the one that tripped
the keyword classifier.


In [ ]:
# ── Live demo: summarize and classify the tricky transcript ────────────────────

TRANSCRIPT = (
    "I don't want a refund — I just need you to fix the broken tracking "
    "on my package. The delivery estimate keeps changing and I'm getting "
    "error messages every time I log in to check my order status. "
    "I placed this order two weeks ago and it still shows as processing. "
    "Can someone please look into this?"
)

print("Transcript:")
print(f"  {TRANSCRIPT}")
print()

key_points = extract_key_points(TRANSCRIPT)
intent     = classify_intent(TRANSCRIPT)

print("Key points:")
print(f"  {key_points}")
print()
print(f"Classified intent: {intent}")
print("  -> Model captures negation ('don't want a refund') — keyword counter could not.")


In [ ]:
# ── Your turn — change the prompt prefix and observe the shift ─────────────────
# CHANGE: swap PROMPT_STYLE below and re-run.
#    "concise"   -> ask for a one-sentence summary
#    "formal"    -> ask for formal bullet points
#    "questions" -> ask for open questions raised by the conversation

PROMPT_STYLE = "concise"   # try: "formal", "questions"

PROMPT_MAP = {
    "concise":   "Summarize this conversation in one sentence: ",
    "formal":    "List formal action items from this conversation: ",
    "questions": "What open questions are raised by this conversation: ",
}

prompt = PROMPT_MAP[PROMPT_STYLE] + TRANSCRIPT
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
result = tokenizer.decode(out[0], skip_special_tokens=True)

print(f"Prompt style: {PROMPT_STYLE}")
print(f"  -> Output: {result}")
print("  -> Same model, different prefix -> different output genre.")


---

## Summary

| Step | Concept | Key Idea |
|------|---------|---------|
| 1 | Why automated analysis? | Keyword counting misses negation and paraphrase |
| 2 | FLAN-T5 architecture | Encoder reads full input; decoder generates conditioned on it |
| 3 | Summarization pipeline | Prompt prefix steers the same model to different output styles |
| 4 | Live demo | Model correctly resolves "don't want a refund" as shipping/technical |
| 5 | Your turn | Prompt prefix is the only knob needed to change task behavior |

**Key insights to keep**

- FLAN-T5 is prompt-driven: the task is encoded in a plain-English prefix, not in model weights.
- Encoder-decoder beats decoder-only for summarization because the encoder sees the full input bidirectionally before generating a single output token.
- For long transcripts, chunk-then-merge is a practical workaround, but cross-chunk context can be lost at seams.
- The same 77 M-parameter model handles summarization, classification, and QA without any fine-tuning — instruction tuning is the key.
